# Notebook 07 — Multi-objective optimization: the binding / developability trade-off

**Where this sits.** Notebook 05 ran the RL loop on developability (three sequence oracles) and it stayed **flat** — the pool was already saturated. Notebook 06 ran the *same* loop on binding and it **rose** 0.30 → 0.80 — binding had headroom. Each was a controlled, single-axis experiment.

This notebook answers the question those two raise together:

> **Can the four objectives be optimized jointly, and what does the trade-off between them look like?**

Binding and developability are different *kinds* of good. A sequence can bind HER2 strongly and still be a poor drug (aggregation-prone, immunogenic), or be beautifully developable and not bind at all. When objectives compete, there is no single "best" sequence — there is a **Pareto front**: the set of designs where you cannot improve one objective without sacrificing another.

**What we do here (post-hoc Pareto analysis).** Rather than re-run RL with a blended reward, we take the 120 loops the generator already produced, score all four objectives on each, and characterize the front. This is the honest first step: it *measures the tension* before committing to a scalarization. A weighted-sum RL run would bake in one particular trade-off; the front shows us the whole menu first.


## 1. The four objectives

All four are oracles that map a CDRH3 to a score in [0, 1], all **maximized**:

| Objective | Oracle | Notebook | What it rewards |
|---|---|---|---|
| **binding** | ESM2-8M + LogReg | 06 | predicted P(binder) to HER2 |
| **humanness** | AbLang2 likelihood | 02 | residues look human (low immunogenicity) |
| **liability** | motif regex | 02 | absence of chemical liability motifs |
| **physchem** | ProtParam | 02 | charge + hydrophobicity in a safe window |

The first three developability oracles were combined in notebooks 04-05 into a single scalar `S(x) = 0.5·humanness + 0.3·liability + 0.2·physchem`. Here we keep them **separate** for binding, and also use the combined `S(x)` as the developability axis for the 2D view.

**Design note (interview-relevant).** We deliberately do NOT normalize the four onto a common scale by z-scoring. Each oracle is already in [0,1] by construction, and Pareto dominance is **scale-invariant to monotone transforms per axis** — it only depends on the ordering within each objective, so no normalization is needed for the front itself. Normalization would only matter if we collapsed to a weighted sum.


In [ ]:
# --- Load the four objectives, pre-computed on the 120 generated loops ---
# (binding from notebook 06's ESM oracle; the three developability oracles from notebook 02.)
import json, numpy as np, matplotlib.pyplot as plt

d = json.load(open("pareto_data.json"))
cdrs   = d["cdrs"]
binding = np.array(d["binding"])   # ESM2 P(HER2 binder)
human   = np.array(d["human"])     # AbLang2 humanness
liab    = np.array(d["liab"])      # liability score
phys    = np.array(d["phys"])      # physicochemical score
dev_S   = np.array(d["dev_S"])     # 0.5*human + 0.3*liab + 0.2*phys  (the notebook-04 reward)
n = len(cdrs)

print(f"pool: {n} loops")
for nm, v in [("binding",binding),("humanness",human),("liability",liab),("physchem",phys)]:
    print(f"  {nm:10s} mean {v.mean():.3f}  std {v.std():.3f}  range {v.min():.2f}-{v.max():.2f}")


## 2. Why the objectives compete — the correlation

If binding and developability were positively correlated, there would be no trade-off: optimizing one would give the other for free. The interesting (and realistic) case is when they are **uncorrelated or opposed**.


In [ ]:
# --- Is there tension? Correlate binding against developability ---
r = np.corrcoef(binding, dev_S)[0,1]
print(f"corr(binding, dev_S) = {r:.3f}")
# best-per-objective: does the champion of one objective win the others?
O = np.column_stack([binding, human, liab, phys]); names=["binding","humanness","liability","physchem"]
for k, nm in enumerate(names):
    i = int(np.argmax(O[:,k]))
    print(f"best {nm:10s}: {cdrs[i]:20s} | binding {binding[i]:.2f} human {human[i]:.2f} liab {liab[i]:.2f} phys {phys[i]:.2f}")


**Reading the output.** The correlation is ≈ 0.07 — essentially **orthogonal**. This is the whole justification for a multi-objective treatment: binding and developability carry independent information, so you cannot get both by optimizing one.

The per-objective champions make it concrete: the **best binder** (P≈0.91) has only middling humanness (~0.63), while the sequences that top humanness/liability/physchem have **near-zero binding** (~0.05). No sequence wins everything. That is the definition of a trade-off, and it is why we need a front, not a ranking.


## 3. The Pareto front

A design is **dominated** if some other design is at least as good on every objective and strictly better on at least one. The **Pareto front** is the set of non-dominated designs — the efficient frontier of the trade-off.

We compute it two ways:
- **2D** (binding vs the combined developability `S(x)`) — easy to plot and read.
- **4D** (all four objectives separately) — the honest full picture; a design only needs to be non-dominated across all four to make the set.


In [ ]:
# --- Pareto dominance (maximization on every axis) ---
def pareto_mask(M):
    """Return boolean mask: True where the row is non-dominated. M is (n, k), all maximized."""
    n = len(M); keep = np.ones(n, bool)
    for i in range(n):
        # a point j dominates i if j >= i on ALL objectives and j > i on AT LEAST ONE
        dominates_i = np.all(M >= M[i], axis=1) & np.any(M > M[i], axis=1)
        if dominates_i.any():
            keep[i] = False
    return keep

front2 = pareto_mask(np.column_stack([binding, dev_S]))   # 2D
front4 = pareto_mask(np.column_stack([binding, human, liab, phys]))  # 4D
print(f"2D Pareto front (binding vs dev_S): {front2.sum()}/{n}")
print(f"4D Pareto front (all four separate): {front4.sum()}/{n}")

# list the 2D front, sorted along the trade-off
idx = np.where(front2)[0]; idx = idx[np.argsort(dev_S[idx])]
print("\n2D front (the efficient trade-off curve):")
for i in idx:
    print(f"  {cdrs[i]:20s} dev_S {dev_S[i]:.3f}  binding {binding[i]:.3f}")


**Reading the output.** Six designs sit on the 2D front; seventeen on the 4D front (more, because extra objectives give more ways to be non-dominated). The 2D front is a clean **downward-sloping curve**: as developability rises from 0.74 to 0.86, the best achievable binding falls from 0.91 to 0.61. That descending shape **is** the trade-off — you buy developability with binding and vice-versa.

The endpoints are the extremes (max binder / max developability); the interior points are **compromise designs**. Which one you pick depends on the program's priority — there is no algorithmic "best", which is exactly the point of showing the front instead of a single number.


In [ ]:
# --- Visualize: the 2D front + a parallel-coordinates view of the 4D set ---
fig,(ax1,ax2)=plt.subplots(1,2,figsize=(11,4.2))

# LEFT: scatter of all loops; Pareto front highlighted and connected
dom=~front2
ax1.scatter(dev_S[dom], binding[dom], s=22, c="#c9c9c9", label="dominated", zorder=2)
ax1.scatter(dev_S[front2], binding[front2], s=55, c="#b2182b", edgecolor="k", lw=0.5, label="Pareto front", zorder=3)
o=np.argsort(dev_S[front2]); ax1.plot(dev_S[front2][o], binding[front2][o], "--", c="#b2182b", lw=1.2, zorder=2)
ax1.set_xlabel("developability  S(x)"); ax1.set_ylabel("binding  P(HER2)")
ax1.set_title(f"Trade-off front (r={r:.2f})"); ax1.legend(fontsize=8, loc="upper left")

# RIGHT: parallel coordinates - each line is one loop across the 4 objectives
xs=np.arange(4)
for i in np.where(~front4)[0]: ax2.plot(xs, O[i], color="#dddddd", lw=0.4, alpha=0.5, zorder=1)
for i in np.where(front4)[0]:  ax2.plot(xs, O[i], color="#2166ac", lw=1.0, alpha=0.7, zorder=2)
ax2.set_xticks(xs); ax2.set_xticklabels(["binding","human","liab","phys"]); ax2.set_ylim(0,1.02)
ax2.set_ylabel("oracle score"); ax2.set_title(f"4-objective Pareto set ({int(front4.sum())}/{n})")
fig.tight_layout(); fig.savefig("nb07_pareto.png", dpi=130, bbox_inches="tight"); plt.show()


## 4. Picking a design from the front — the knee point

When there is no external priority, a common heuristic is the **knee**: the front point that maximizes the product of objectives (balanced improvement), i.e. the point of diminishing returns where sacrificing a little of one buys little of the other.


In [ ]:
# --- The knee: balanced compromise (max binding * dev_S on the front) ---
fi = np.where(front2)[0]
knee = fi[np.argmax((binding*dev_S)[fi])]
print(f"knee design: {cdrs[knee]}")
print(f"  binding {binding[knee]:.3f}  dev_S {dev_S[knee]:.3f}  "
      f"(human {human[knee]:.2f} liab {liab[knee]:.2f} phys {phys[knee]:.2f})")
print("\nInterpretation: this is the single sequence to advance if forced to choose one,")
print("balancing predicted binding against developability without committing to either extreme.")


**Reading the output.** The knee is `ALFDYGVHDGYFDY` — it happens to also be the strongest binder (P≈0.91) while still holding a respectable developability (S≈0.74). In this particular pool the binding axis has so much more spread than developability that the knee coincides with the binding champion; with a denser front the knee would typically be an interior compromise.

**Caveat.** The knee is a heuristic, not a decision. A real program would weight the objectives by context (e.g. immunogenicity is a hard gate for a therapeutic, so humanness might be a constraint rather than an objective).


## 5. What this adds, and its honest limits

**What it adds.**
1. It turns the two single-axis RL runs (nb05 flat, nb06 rising) into one coherent picture: the axes are **orthogonal** (r≈0.07), so they genuinely trade off, and the front quantifies the exchange rate.
2. It produces **actionable output**: a shortlist of non-dominated designs and a principled way (knee) to pick among them.

**Honest limits.**
- This is **post-hoc** on a fixed pool, not multi-objective *optimization*. We characterize the trade-off among sequences the generator already made; we do not push the generator toward the front. The natural next step is a scalarized RL run (weighted sum, or a constraint like "maximize binding s.t. developability ≥ 0.8") — the front tells us where to set that constraint.
- Everything downstream of binding is a **predicted** score under a transferred proxy oracle, not measured affinity (see notebook 06). The claim is about the *structure of the trade-off*, not about validated binders.
- The front is only as good as the pool's diversity. 120 loops is enough to see the shape; a production run would generate thousands.

**One-line takeaway.** Binding and developability are orthogonal objectives; the Pareto front makes their trade-off explicit and yields a defensible shortlist — the missing multi-objective piece that ties the RL story together.
